# LLM 流式输出：跨 Chunk Stop Sequence 的安全提交

**面试问题：Stop Sequence 可能横跨 SSE Chunk 时，怎样避免把停止标记或其后内容先发给用户？**

## 回答主线

1. 流式服务拿到的网络 Chunk 边界与 Token、字符和 Stop Sequence 边界没有任何保证。
2. 每个 Chunk 到达就原样发送，会在 stop 横跨两块时先泄露标记前缀，之后无法从客户端撤回。
3. 安全提交器必须保留最长 stop 长度减一的未决后缀，只发送不可能再组成 stop 的前缀。
4. 一旦在缓冲区发现最早 stop，只提交其前内容并丢弃 stop 及之后内容。
5. 取消和连接断开时不能自动 flush 未决后缀，否则仍可能泄露半个控制标记。
6. 多 stop、Unicode 字节解码和 backpressure 都应进入状态机测试。

## 真实案例

五条客服流使用 `<END>` 作为停止标记，但服务端分别把标记切在一个、两个或三个 Chunk 中，另有无标记正常结束和标记后跟内部草稿的样本。我们对比立即推送与带后缀保留的安全提交账本，再复现取消时错误 flush。这是可读的离线教学实验，用来验证协议和算法，不能外推为线上模型收益。

### 输入预览：五种真实 Chunk 切分

In [1]:
streams = [  # 构造五种停止标记边界。
    {"id": "S1", "chunks": ["退款将在3天到账", "<END>"], "expected": "退款将在3天到账"},  # stop 完整落在第二块。
    {"id": "S2", "chunks": ["订单已取消<EN", "D>内部草稿"], "expected": "订单已取消"},  # stop 横跨两块且后有敏感草稿。
    {"id": "S3", "chunks": ["验证码已发送<", "EN", "D>勿展示"], "expected": "验证码已发送"},  # stop 横跨三块。
    {"id": "S4", "chunks": ["包裹", "明天", "送达"], "expected": "包裹明天送达"},  # 无 stop 时正常结束需要 flush。
    {"id": "S5", "chunks": ["发票已", "发送<END>trace=abc"], "expected": "发票已发送"},  # stop 和内部 trace 位于同一块。
]  # 完成流式样本。
stop_sequences = ["<END>"]  # 定义当前协议的停止标记集合。
print("流  chunks                                  expected")  # 输出输入表头。
for stream in streams:  # 逐流展示网络切分方式。
    print(f"{stream['id']}  {stream['chunks']} -> {stream['expected']}")  # 展示跨块 stop 场景。

流  chunks                                  expected
S1  ['退款将在3天到账', '<END>'] -> 退款将在3天到账
S2  ['订单已取消<EN', 'D>内部草稿'] -> 订单已取消
S3  ['验证码已发送<', 'EN', 'D>勿展示'] -> 验证码已发送
S4  ['包裹', '明天', '送达'] -> 包裹明天送达
S5  ['发票已', '发送<END>trace=abc'] -> 发票已发送


## Baseline 基线：Chunk 到达即发送再搜索 Stop

In [2]:
def eager_stream(chunks, stop):  # 实现无法撤回的立即推送基线。
    sent = ""  # 收集已经暴露给客户端的内容。
    joined = ""  # 收集服务端已见完整文本用于检测。
    events = []  # 保存每块发送和检测事件。
    for chunk in chunks:  # 按网络到达顺序处理 Chunk。
        sent += chunk  # 错误地先把整块提交给客户端。
        joined += chunk  # 再把 Chunk 加入检测缓冲。
        detected = stop in joined  # 检查累计文本是否终于出现 stop。
        events.append({"chunk": chunk, "sent_so_far": sent, "detected": detected})  # 记录不可逆发送状态。
        if detected:  # stop 被发现后停止读取。
            break  # 但已经发送的标记和后续文字无法撤回。
    return sent, events  # 返回实际泄露文本和事件。

baseline_rows = []  # 收集五条流的基线结果。
for stream in streams:  # 逐流运行立即推送。
    sent, events = eager_stream(stream["chunks"], "<END>")  # 执行基线状态机。
    baseline_rows.append({"id": stream["id"], "sent": sent, "correct": sent == stream["expected"], "events": events})  # 保存结果。
print("流  基线发送内容                              正确")  # 输出基线表头。
for row in baseline_rows:  # 逐流展示不可撤回内容。
    print(f"{row['id']}  {row['sent']:<38} {row['correct']}")  # 展示 stop 和内部草稿泄露。

流  基线发送内容                              正确
S1  退款将在3天到账<END>                          False
S2  订单已取消<END>内部草稿                         False
S3  验证码已发送<END>勿展示                         False
S4  包裹明天送达                                 True
S5  发票已发送<END>trace=abc                    False


### 核心实现：最长后缀保留与最早 Stop 检测

In [3]:
class SafeStopStreamer:  # 实现跨 Chunk 的安全提交状态机。
    def __init__(self, stops):  # 初始化多个停止序列。
        self.stops = list(stops)  # 保存 stop 集合。
        self.holdback = max(len(stop) for stop in stops) - 1  # 计算必须保留的最大未决后缀长度。
        self.buffer = ""  # 保存尚未安全提交的文本。
        self.committed = ""  # 保存已经发送给客户端的文本。
        self.stopped = False  # 记录是否已命中 stop。
        self.events = []  # 保存每次 feed 的状态变化。

    def feed(self, chunk):  # 接收一个网络 Chunk 并返回本次安全可发文本。
        if self.stopped:  # stop 后到达的 Chunk 必须忽略。
            self.events.append({"chunk": chunk, "emit": "", "buffer": self.buffer, "state": "ignored-after-stop"})  # 记录被忽略内容。
            return ""  # 不再向客户端发送。
        self.buffer += chunk  # 将新块追加到未决缓冲。
        matches = [(self.buffer.find(stop), stop) for stop in self.stops if stop in self.buffer]  # 搜索当前缓冲内所有完整 stop。
        if matches:  # 至少一个 stop 已经完整出现。
            position, matched_stop = min(matches, key=lambda item: item[0])  # 选择文本中最早出现的 stop。
            emitted = self.buffer[:position]  # 只提交 stop 之前的正文。
            self.committed += emitted  # 更新客户端已见内容。
            self.buffer = ""  # 丢弃 stop 及其后的内部内容。
            self.stopped = True  # 进入终止状态。
            self.events.append({"chunk": chunk, "emit": emitted, "buffer": self.buffer, "state": f"stopped:{matched_stop}"})  # 记录命中事件。
            return emitted  # 返回本次安全正文。
        safe_length = max(0, len(self.buffer) - self.holdback)  # 只提交不可能成为 stop 前缀的部分。
        emitted = self.buffer[:safe_length]  # 提取安全前缀。
        self.buffer = self.buffer[safe_length:]  # 保留最多 stop 长度减一的后缀。
        self.committed += emitted  # 更新已发送正文。
        self.events.append({"chunk": chunk, "emit": emitted, "buffer": self.buffer, "state": "open"})  # 记录缓冲变化。
        return emitted  # 返回当前可安全发送内容。

    def finish(self):  # 在上游正常结束且未命中 stop 时提交剩余正文。
        if self.stopped:  # 已命中 stop 时不应再 flush。
            return ""  # 返回空文本。
        emitted = self.buffer  # 正常 EOF 证明后缀不会再组成 stop。
        self.buffer = ""  # 清空缓冲。
        self.committed += emitted  # 提交最后正文。
        self.events.append({"chunk": "<EOF>", "emit": emitted, "buffer": "", "state": "finished"})  # 记录正常结束。
        return emitted  # 返回最后可发内容。

demo = SafeStopStreamer(stop_sequences)  # 创建跨三块样本的状态机。
for chunk in streams[2]["chunks"]:  # 逐块喂入 S3。
    demo.feed(chunk)  # 执行安全提交。
demo.finish()  # stop 已命中时 finish 不再发送。
print("S3 安全提交账本：")  # 输出最复杂跨块案例。
for event in demo.events:  # 逐事件展示 emit 和 holdback。
    print(event)  # 展示 `<`、`EN` 如何被保留直到组成完整 stop。

S3 安全提交账本：
{'chunk': '验证码已发送<', 'emit': '验证码', 'buffer': '已发送<', 'state': 'open'}
{'chunk': 'EN', 'emit': '已发', 'buffer': '送<EN', 'state': 'open'}
{'chunk': 'D>勿展示', 'emit': '送', 'buffer': '', 'state': 'stopped:<END>'}


## 结果解读：逐流对照实际客户端可见内容

In [4]:
safe_rows = []  # 收集安全提交结果。
for stream in streams:  # 逐流创建独立状态机。
    streamer = SafeStopStreamer(stop_sequences)  # 初始化干净缓冲和状态。
    for chunk in stream["chunks"]:  # 按原网络边界输入。
        streamer.feed(chunk)  # 只提交安全前缀。
    streamer.finish()  # 对没有 stop 的 S4 正常 flush。
    safe_rows.append({"id": stream["id"], "sent": streamer.committed, "correct": streamer.committed == stream["expected"], "events": streamer.events})  # 保存客户端真实可见内容。
print("流  Baseline正确  Safe正确  Safe发送内容")  # 输出同口径结果表头。
for baseline, safe in zip(baseline_rows, safe_rows):  # 对齐两种状态机。
    print(f"{safe['id']}  {str(baseline['correct']):<12} {str(safe['correct']):<9} {safe['sent']}")  # 展示五条安全结果。
baseline_accuracy = sum(row["correct"] for row in baseline_rows) / len(streams)  # 计算立即推送正确率。
safe_accuracy = sum(row["correct"] for row in safe_rows) / len(streams)  # 计算安全提交正确率。
print(f"正确率 {baseline_accuracy:.0%} -> {safe_accuracy:.0%}，最大额外字符延迟={max(len(stop) for stop in stop_sequences) - 1}")  # 报告正确性和延迟代价。
print("解读：安全性来自最多保留4个字符；无 stop 的 S4 只在正常 EOF 时发送未决后缀。")  # 解释算法代价。

流  Baseline正确  Safe正确  Safe发送内容
S1  False        True      退款将在3天到账
S2  False        True      订单已取消
S3  False        True      验证码已发送
S4  True         True      包裹明天送达
S5  False        True      发票已发送
正确率 20% -> 100%，最大额外字符延迟=4
解读：安全性来自最多保留4个字符；无 stop 的 S4 只在正常 EOF 时发送未决后缀。


## 失败案例：连接取消时错误 Flush 未决控制前缀

In [5]:
cancel_streamer = SafeStopStreamer(stop_sequences)  # 创建即将被取消的流。
cancel_streamer.feed("操作失败<EN")  # 输入正文和 stop 的不完整前缀。
unsafe_cancel_output = cancel_streamer.committed + cancel_streamer.buffer  # 模拟取消处理器错误 flush 全部缓冲。
safe_cancel_output = cancel_streamer.committed  # 正确取消只保留此前已经安全提交的正文。
print(f"取消前 committed={cancel_streamer.committed!r} holdback={cancel_streamer.buffer!r}")  # 展示未决 `<EN` 仍在服务端。
print(f"错误 flush 客户端看到={unsafe_cancel_output!r}")  # 展示半个控制标记泄露。
print(f"安全取消客户端看到={safe_cancel_output!r}")  # 展示丢弃未决后缀后的正文。
print("修正策略：区分 normal_eof、stop、cancel、error；只有 normal_eof 能 flush，其他终态丢弃 holdback。")  # 总结终态语义。

取消前 committed='操作失' holdback='败<EN'
错误 flush 客户端看到='操作失败<EN'
安全取消客户端看到='操作失'
修正策略：区分 normal_eof、stop、cancel、error；只有 normal_eof 能 flush，其他终态丢弃 holdback。


### 生产边界与 SSE 事件

In [6]:
sse_event = {"request_id": "S2", "event": "stop_detected", "stop": "<END>", "committed_chars": len(safe_rows[1]["sent"]), "discarded_after_stop": True, "algorithm": "max-stop-suffix-r1"}  # 构造流式审计事件。
print("SSE 事件：", sse_event)  # 展示排查泄露所需字段。
print("生产替换点：真实服务还需 UTF-8 增量解码、Token stop、多个重叠 stop、SSE resume id、backpressure、取消传播和网关缓冲测试。")  # 明确字符教学状态机边界。

SSE 事件： {'request_id': 'S2', 'event': 'stop_detected', 'stop': '<END>', 'committed_chars': 5, 'discarded_after_stop': True, 'algorithm': 'max-stop-suffix-r1'}
生产替换点：真实服务还需 UTF-8 增量解码、Token stop、多个重叠 stop、SSE resume id、backpressure、取消传播和网关缓冲测试。


## 回归测试：最后只保护跨块、EOF 与取消语义

In [7]:
assert safe_accuracy == 1.0 and safe_accuracy > baseline_accuracy  # 验证五种切分均被安全状态机正确处理。
assert safe_rows[1]["sent"] == "订单已取消" and "内部草稿" not in safe_rows[1]["sent"]  # 验证跨两块 stop 不泄露后续内容。
assert safe_rows[2]["sent"] == "验证码已发送"  # 验证跨三块 stop 正确终止。
assert safe_rows[3]["sent"] == streams[3]["expected"] and not safe_rows[3]["events"][-1]["state"].startswith("stopped")  # 验证无 stop 的正常 EOF 完整 flush。
assert unsafe_cancel_output.endswith("<EN") and "<" not in safe_cancel_output and safe_cancel_output == cancel_streamer.committed  # 验证取消时宁可丢弃未决正文后缀也不泄露控制标记。
print("回归测试通过：两块/三块 Stop、后续丢弃、正常 EOF 和取消 Holdback 均成立。")  # 用少量断言总结流式合同。

回归测试通过：两块/三块 Stop、后续丢弃、正常 EOF 和取消 Holdback 均成立。
